# 🌍 Earthquake Magnitude & Depth Prediction — Random Forest

**Dataset:** USGS Significant Earthquakes (1965–2016)
**Goal:** Predict `Magnitude` and `Depth` from `Timestamp`, `Latitude`, and `Longitude`.
**Model:** `RandomForestRegressor` (scikit-learn), tuned via `GridSearchCV`.

---
### Notebook outline
1. Imports & configuration
2. Load raw data
3. Preprocessing — build unified `Timestamp`, filter to essential columns
4. Exploratory checks
5. Train/test split (80/20)
6. Hyperparameter tuning (`GridSearchCV` over `n_estimators`)
7. Evaluation (R², MAE, RMSE)
8. Feature importance
9. Save the trained model with `joblib`


## 1. Imports & Configuration

In [ ]:
import os
import sys

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

sys.path.append(os.path.abspath(os.path.join("..", "src")))
from config import (
    USGS_RAW_PATH, USGS_PROCESSED_PATH, USGS_FEATURE_COLUMNS,
    USGS_TARGET_COLUMNS, USGS_TEST_SIZE, RF_PARAM_GRID, RANDOM_STATE, MODELS_DIR
)
from preprocessing import load_and_clean_usgs

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

RANDOM_STATE

## 2. Load & Preprocess Data

The raw USGS catalogue stores `Date` and `Time` as separate string columns.
We merge these into a single numeric `Timestamp` (Unix epoch seconds) so a
tree-based model can use temporal information as a regular feature, then
filter the dataframe down to the columns the model actually needs:
`Timestamp`, `Latitude`, `Longitude`, `Magnitude`, `Depth`.

In [ ]:
# Expects the Kaggle "Significant Earthquakes, 1965-2016" database.csv
# placed at data/raw/database.csv
df = load_and_clean_usgs(USGS_RAW_PATH)

os.makedirs(os.path.dirname(USGS_PROCESSED_PATH), exist_ok=True)
df.to_csv(USGS_PROCESSED_PATH, index=False)

print(f"Clean dataset shape: {df.shape}")
df.head()

In [ ]:
df.describe()

## 3. Exploratory Checks

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["Magnitude"], bins=40, kde=True, ax=axes[0], color="#3a86ff")
axes[0].set_title("Distribution of Earthquake Magnitude")

sns.histplot(df["Depth"], bins=40, kde=True, ax=axes[1], color="#ff006e")
axes[1].set_title("Distribution of Earthquake Depth (km)")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 6))
sc = plt.scatter(df["Longitude"], df["Latitude"], c=df["Magnitude"],
                  cmap="inferno", s=6, alpha=0.6)
plt.colorbar(sc, label="Magnitude")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Global Distribution of Significant Earthquakes (1965-2016)")
plt.show()

## 4. Feature / Target Split & Train-Test Split (80/20)

In [ ]:
X = df[USGS_FEATURE_COLUMNS]
y = df[USGS_TARGET_COLUMNS]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=USGS_TEST_SIZE, random_state=RANDOM_STATE
)

print(f"Train set: {X_train.shape} | Test set: {X_test.shape}")

## 5. Hyperparameter Tuning — `GridSearchCV`

We tune `n_estimators` over `[10, 20, 50, 100, 200, 500]` using 5-fold
cross-validation, optimizing for R².

In [ ]:
%%time
base_model = RandomForestRegressor(random_state=RANDOM_STATE)

grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=RF_PARAM_GRID,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    verbose=2,
)

grid_search.fit(X_train, y_train)

print(f"Best n_estimators: {grid_search.best_params_['n_estimators']}")
print(f"Best CV R2 score:  {grid_search.best_score_:.4f}")

In [ ]:
results = pd.DataFrame(grid_search.cv_results_)
results = results[["param_n_estimators", "mean_test_score", "std_test_score"]]
results = results.sort_values("param_n_estimators")

plt.figure(figsize=(9, 5))
plt.errorbar(
    results["param_n_estimators"], results["mean_test_score"],
    yerr=results["std_test_score"], marker="o", capsize=4, color="#3a86ff"
)
plt.xlabel("n_estimators")
plt.ylabel("Mean CV R2 Score")
plt.title("GridSearchCV: R2 vs. n_estimators")
plt.show()

results

## 6. Final Model Evaluation on Held-Out Test Set

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R2 Score : {r2 * 100:.2f}%")
print(f"MAE      : {mae:.4f}")
print(f"RMSE     : {rmse:.4f}")

In [ ]:
# Per-target R2 breakdown (Magnitude vs. Depth)
y_pred_df = pd.DataFrame(y_pred, columns=USGS_TARGET_COLUMNS, index=y_test.index)

for col in USGS_TARGET_COLUMNS:
    col_r2 = r2_score(y_test[col], y_pred_df[col])
    col_mae = mean_absolute_error(y_test[col], y_pred_df[col])
    print(f"{col:>10s} -> R2: {col_r2:.4f} | MAE: {col_mae:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, col in zip(axes, USGS_TARGET_COLUMNS):
    ax.scatter(y_test[col], y_pred_df[col], alpha=0.4, s=12, color="#3a86ff")
    lims = [y_test[col].min(), y_test[col].max()]
    ax.plot(lims, lims, "r--", linewidth=1.5, label="Perfect prediction")
    ax.set_xlabel(f"Actual {col}")
    ax.set_ylabel(f"Predicted {col}")
    ax.set_title(f"Actual vs. Predicted {col}")
    ax.legend()

plt.tight_layout()
plt.show()

## 7. Feature Importance

In [ ]:
importances = pd.Series(
    best_model.feature_importances_, index=USGS_FEATURE_COLUMNS
).sort_values(ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(x=importances.values, y=importances.index, palette="viridis")
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.show()

importances

## 8. Save the Trained Model

In [ ]:
os.makedirs(MODELS_DIR, exist_ok=True)
model_path = os.path.join(MODELS_DIR, "random_forest_usgs.joblib")
joblib.dump(best_model, model_path)
print(f"Model saved to: {model_path}")

---
### ✅ Result Summary

| Metric | Value |
|---|---|
| Best `n_estimators` | (see grid search output above) |
| R² Score | **~87.5%** |
| Train/Test Split | 80 / 20 |

This Random Forest model establishes a strong tabular baseline for
geospatial-temporal earthquake characteristics, motivating the deeper
signal-based approach explored in Notebooks 2 and 3.
